
# ProofLaunch Radeon Gate V6

**Track 1 — Development of Multimodal Content Creation Tools**

Run only when the kernel shows **Connected**. This gate fails closed. It does
not publish, submit a PR, overwrite the Gemma training Notebook, or reinterpret
the procedural fallback as model output.


In [ ]:

from pathlib import Path
import hashlib, json, os, platform, subprocess, sys, time

print("python:", sys.version)
print("platform:", platform.platform())
print("cwd:", Path.cwd())


## 1. Reassemble, hash-verify, and extract the versioned source


In [ ]:

archive = Path("prooflaunch-radeon-v6.tar.gz")
parts = sorted(Path(".").glob("prooflaunch-radeon-v6.part-*"))
if not archive.exists() and parts:
    with archive.open("wb") as destination:
        for part in parts:
            destination.write(part.read_bytes())
    print(f"reassembled {len(parts)} parts -> {archive}")

checksum_file = Path("prooflaunch-radeon-v6.sha256")
assert archive.is_file(), f"Missing archive and parts: {archive}"
assert checksum_file.is_file(), f"Missing checksum: {checksum_file}"
expected = checksum_file.read_text().split()[0].strip()
actual = hashlib.sha256(archive.read_bytes()).hexdigest()
print("archive sha256:", actual)
assert actual == expected, f"Archive hash mismatch: expected {expected}, got {actual}"
subprocess.run(["tar", "-xzf", str(archive)], check=True)
repo = Path("prooflaunch_submission")
assert (repo / "pyproject.toml").is_file(), "Extracted repository is incomplete"


## 2. Install declared dependencies and run Level-1 gates


In [ ]:

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{repo}[radeon]"],
    check=True,
)
env = {**os.environ, "PYTHONPATH": str(repo / "src")}
tests = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", str(repo / "tests"), "-v"],
    env=env, text=True, capture_output=True
)
print(tests.stdout)
print(tests.stderr)
assert tests.returncode == 0, "Unit/integration tests failed"

control_eval = subprocess.run(
    [
        sys.executable,
        str(repo / "scripts" / "run_control_plane_eval.py"),
        "--cases-per-mode", "50",
        "--output", str(repo / "evidence" / "radeon" / "control-plane-eval.json"),
    ],
    env=env, text=True, capture_output=True
)
print(control_eval.stdout)
print(control_eval.stderr)
assert control_eval.returncode == 0, "250-case control-plane gate failed"


## 3. Prove the local AMD Radeon runtime


In [ ]:

import torch

runtime = {
    "torch_version": torch.__version__,
    "hip_version": getattr(torch.version, "hip", None),
    "gpu_available": torch.cuda.is_available(),
    "gpu_count": torch.cuda.device_count(),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "total_vram_bytes": (
        torch.cuda.get_device_properties(0).total_memory
        if torch.cuda.is_available() else 0
    ),
}
print(json.dumps(runtime, indent=2))
assert runtime["gpu_available"], "AMD Radeon GPU is not available"
assert runtime["hip_version"], "PyTorch does not report a ROCm/HIP runtime"

from prooflaunch_remix.model_registry import RadeonModelRouter

available_vram_gb = runtime["total_vram_bytes"] / (1024 ** 3)
execution_plan = RadeonModelRouter().report(
    available_vram_gb=available_vram_gb,
    roles=(
        "vision_language_audit",
        "open_set_detection",
        "affect_critic",
        "image_generation",
        "image_text_scoring",
        "video_generation",
        "temporal_segmentation",
        "audio_text_scoring",
    ),
    include_candidates=True,
)
print(json.dumps(execution_plan, indent=2))



## 3A. Build the inspectable multimodal mission before loading any model

This V6 path accepts independently hashed image, PDF, audio, and video inputs.
The frozen demonstration uses an owned product image plus an exact-page PDF
brief, then requests both a social image and a short image-conditioned video.
No free-running swarm may change the graph after its hash is recorded.


In [ ]:

import gc
from PIL import Image, ImageDraw
import fitz

from prooflaunch_remix.mission import (
    DeliverableKind,
    DeliverableSpec,
    MultimodalMissionPlanner,
)
from prooflaunch_remix.quality_contract import QualityTier
from prooflaunch_remix.multimodal_ingest import (
    MediaSource,
    MultimodalIngestor,
    RightsStatus,
)

multimodal_run_dir = repo / "evidence" / "radeon" / "multimodal-v6"
multimodal_run_dir.mkdir(parents=True, exist_ok=True)
mission_product_path = multimodal_run_dir / "owned-product.png"
mission_product = Image.new("RGBA", (720, 900), (0, 0, 0, 0))
mission_draw = ImageDraw.Draw(mission_product)
mission_draw.rounded_rectangle(
    (95, 55, 625, 835), radius=90,
    fill=(245, 248, 255, 255), outline=(174, 201, 239, 255), width=12,
)
mission_draw.rounded_rectangle(
    (185, 180, 535, 565), radius=36, fill=(16, 74, 154, 255)
)
mission_draw.ellipse((275, 655, 445, 825), fill=(255, 187, 76, 255))
mission_product.save(mission_product_path, format="PNG", optimize=False)

brief_pdf_path = multimodal_run_dir / "owned-brief.pdf"
brief_document = fitz.open()
brief_page = brief_document.new_page()
brief_page.insert_text(
    (72, 96),
    "Aster Arc evidence brief\nBattery life up to 18 hours under the documented test.\n"
    "Market: Japan. Platform: TikTok. Target emotion: joy.",
)
brief_document.save(brief_pdf_path)
brief_document.close()

mission_artifacts = MultimodalIngestor().ingest(
    (
        MediaSource(
            str(mission_product_path),
            "product_identity",
            RightsStatus.OWNED,
        ),
        MediaSource(
            str(brief_pdf_path),
            "evidence_brief",
            RightsStatus.OWNED,
        ),
    )
)
mission_plan = MultimodalMissionPlanner().build(
    mission_id="radeon-multimodal-v6",
    artifacts=mission_artifacts,
    deliverables=(
        DeliverableSpec(DeliverableKind.SOCIAL_IMAGE, "9:16"),
        DeliverableSpec(DeliverableKind.SHORT_VIDEO, "3:2", 4),
    ),
    available_vram_gb=available_vram_gb,
    quality_tier=QualityTier.COMPETITION_2X,
)
mission_bundle = {
    "artifacts": [item.serializable() for item in mission_artifacts],
    "plan": mission_plan.serializable(),
    "graph_hash": mission_plan.graph_hash,
}
(multimodal_run_dir / "mission-plan.json").write_text(
    json.dumps(mission_bundle, indent=2), encoding="utf-8"
)
print(json.dumps(mission_bundle, indent=2))



## 3B. Understand image evidence with local Qwen3-VL; extract PDF text exactly

Qwen3-VL must return source IDs and spatial/frame locators. The PDF extractor
keeps exact page text rather than asking a model to paraphrase evidence. Both
adapters fail closed on missing locators or invented source IDs.


In [ ]:

from prooflaunch_remix.multimodal_models import (
    LocalPDFTextExtractor,
    LocalQwen3VLAnalyzer,
)
from prooflaunch_remix.multimodal_ingest import MediaKind

image_artifacts = tuple(
    item for item in mission_artifacts if item.kind is MediaKind.IMAGE
)
pdf_artifact = next(
    item for item in mission_artifacts if item.kind is MediaKind.PDF
)
qwen_analyzer = LocalQwen3VLAnalyzer(
    model_id="Qwen/Qwen3-VL-2B-Instruct",
    max_new_tokens=512,
)
qwen_understanding = qwen_analyzer.analyze(
    image_artifacts,
    mission_prompt=(
        "Describe only visible product identity, colors, shape, logo/text, "
        "and composition constraints for an evidence-grounded launch."
    ),
)
pdf_observations = LocalPDFTextExtractor().extract(pdf_artifact)
understanding_evidence = {
    "qwen": {
        "summary": qwen_understanding.summary,
        "observations": [item.__dict__ for item in qwen_understanding.observations],
        "model_ref": qwen_understanding.model_ref,
    },
    "pdf": [item.__dict__ for item in pdf_observations],
}
(multimodal_run_dir / "understanding.json").write_text(
    json.dumps(understanding_evidence, indent=2), encoding="utf-8"
)
print(json.dumps(understanding_evidence, indent=2))

del qwen_analyzer
gc.collect()
torch.cuda.empty_cache()
print("Qwen3-VL phase complete; Radeon memory released")



## 3C. Ground the product with local GroundingDINO

This is a spatial specialist, not a second general chat model. It converts the
owned product phrase into a scored box and source-located observations. The
selected box is later mapped into the composited video and becomes SAM2's
tracking prompt. The model is unloaded before Gemma starts.


In [ ]:

from prooflaunch_remix.multimodal_models import LocalGroundingDINO

product_artifact = image_artifacts[0]
dino = LocalGroundingDINO(
    model_id="IDEA-Research/grounding-dino-tiny",
    box_threshold=0.30,
    text_threshold=0.25,
)
dino_detections = dino.detect(
    product_artifact,
    text_labels=("electronic product", "electronic device", "product"),
)
dino_detection = max(dino_detections, key=lambda item: item.score)
dino_observations = tuple(
    item.to_observation(index)
    for index, item in enumerate(dino_detections, start=1)
)
dino_evidence = {
    "schema": "ProofLaunchGroundingDINOEvidenceV1",
    "model_ref": dino.model_id,
    "selected": dino_detection.__dict__,
    "detections": [item.__dict__ for item in dino_detections],
    "observations": [item.__dict__ for item in dino_observations],
}
(multimodal_run_dir / "grounding-dino.json").write_text(
    json.dumps(dino_evidence, indent=2), encoding="utf-8"
)
print(json.dumps(dino_evidence, indent=2))

del dino
gc.collect()
torch.cuda.empty_cache()
print("GroundingDINO phase complete; Radeon memory released")



## 4. Reload, evaluate, record, and unload the multimodal-safe Gemma adapter

The gate requires the multimodal-retention artifact produced by
`gemma4_emotion_lora_multimodal_safe_v2.ipynb`. It reloads the local adapter,
runs the frozen emotion set, records the campaign pulse, then unloads Gemma
before SDXL enters Radeon memory.


In [ ]:

import gc, re
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from prooflaunch_remix.audience_pulse import GemmaAudiencePulse, RecordedAudiencePulse
from prooflaunch_remix.contracts import CreativeBrief

base_model_dir = Path("./models/google/gemma-4-E4B-it")
adapter_dir = Path("./gemma4-it-emotion-lora-ms-single-gpu")
retention_path = adapter_dir / "multimodal_retention.json"
assert base_model_dir.is_dir(), f"Missing local Gemma model: {base_model_dir}"
assert adapter_dir.is_dir(), f"Missing safe LoRA adapter: {adapter_dir}"
assert retention_path.is_file(), (
    "Missing multimodal retention evidence; run the safe V2 training Notebook"
)
retention = json.loads(retention_path.read_text(encoding="utf-8"))
assert retention["post_accuracy"] >= retention["minimum_post_accuracy"]
assert (
    retention["pre_accuracy"] - retention["post_accuracy"]
    <= retention["maximum_allowed_drop"]
)

gemma_tokenizer = AutoTokenizer.from_pretrained(
    base_model_dir,
    use_fast=True,
    trust_remote_code=True,
)
gemma_base = AutoModelForCausalLM.from_pretrained(
    base_model_dir,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).to("cuda")
gemma_ft = PeftModel.from_pretrained(gemma_base, adapter_dir).to("cuda")
gemma_ft.eval()
valid_emotions = {"sadness", "joy", "love", "anger", "fear", "surprise"}

def predict_emotion_ft(user_text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "Read the user's text and answer with exactly one label. "
                "Only choose from: sadness, joy, love, anger, fear, surprise."
            ),
        },
        {
            "role": "user",
            "content": f"Classify the emotion of this text:\n\n{user_text}",
        },
    ]
    inputs = gemma_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {key: value.to("cuda") for key, value in inputs.items()}
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        output = gemma_ft.generate(
            **inputs,
            max_new_tokens=4,
            do_sample=False,
            pad_token_id=gemma_tokenizer.pad_token_id,
            eos_token_id=gemma_tokenizer.eos_token_id,
        )
    raw = gemma_tokenizer.decode(
        output[0][input_len:],
        skip_special_tokens=True,
    ).strip().lower()
    match = re.search(r"\b(sadness|joy|love|anger|fear|surprise)\b", raw)
    return match.group(1) if match else "invalid"

audience_pulse = GemmaAudiencePulse(
    classifier=predict_emotion_ft,
    model_ref="google/gemma-4-E4B-it+emotion-lora:multimodal-safe-v2",
)
emotion_cases = [
    ("I feel joyful and bright today.", "joy"),
    ("This wonderful launch makes me happy.", "joy"),
    ("I love spending time with you.", "love"),
    ("My heart is full of affection.", "love"),
    ("I am frightened by the sudden noise.", "fear"),
    ("That shadow makes me afraid.", "fear"),
    ("I am furious about the mistake.", "anger"),
    ("The unfair result makes me angry.", "anger"),
    ("I feel deeply sad about the loss.", "sadness"),
    ("The goodbye left me sorrowful.", "sadness"),
    ("What an unexpected surprise!", "surprise"),
    ("I cannot believe that happened.", "surprise"),
]
emotion_predictions = [
    {"text": text, "expected": expected, "predicted": predict_emotion_ft(text).strip().lower()}
    for text, expected in emotion_cases
]
emotion_accuracy = sum(
    row["expected"] == row["predicted"] for row in emotion_predictions
) / len(emotion_predictions)
emotion_evidence = {
    "model_ref": audience_pulse.model_ref,
    "case_count": len(emotion_predictions),
    "accuracy": emotion_accuracy,
    "required_accuracy": 0.80,
    "predictions": emotion_predictions,
    "limitation": "Six synthetic labels do not measure campaign persuasion.",
}
radeon_evidence = repo / "evidence" / "radeon"
radeon_evidence.mkdir(parents=True, exist_ok=True)
(radeon_evidence / "post-finetune-evaluation.json").write_text(
    json.dumps(emotion_evidence, indent=2), encoding="utf-8"
)
print(json.dumps(emotion_evidence, indent=2))
assert emotion_accuracy >= 0.80, "Post-fine-tuning accuracy is below the precommitted bar"

preflight_brief = CreativeBrief(
    id="BRF-001",
    headline="Aster Arc — Battery life up to 18 hours under the documented test.",
    claim_ids=("CLM-001",),
    visual_direction="evidence-led launch",
    target_emotion="joy",
)
recorded_audience = RecordedAudiencePulse(audience_pulse.review(preflight_brief))

del audience_pulse, predict_emotion_ft, gemma_ft, gemma_base, gemma_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Gemma phase complete; Radeon memory released before image generation")



## 5. Run the key local visual inference on Radeon

The default is SDXL Base through Diffusers. Override `PROOFLAUNCH_IMAGE_MODEL`
only with a locally runnable model whose identifier and license are recorded.
The model creates the background; ProofLaunch composites the immutable product
layer, audits four mechanical gates plus the SigLIP2 semantic gate, repairs at
most twice, then exports.


In [ ]:

from diffusers import AutoPipelineForText2Image
from PIL import Image, ImageDraw

from prooflaunch_remix.contracts import CampaignRequest, Stage
from prooflaunch_remix.files import sha256_file
from prooflaunch_remix.ledger import EvidenceLedger
from prooflaunch_remix.runtime import build_supervisor
from prooflaunch_remix.studio import DiffusersBackgroundProvider
from prooflaunch_remix.visual_semantics import HuggingFaceSigLIP2Scorer

model_id = os.environ.get(
    "PROOFLAUNCH_IMAGE_MODEL",
    "stabilityai/stable-diffusion-xl-base-1.0",
)
steps = int(os.environ.get("PROOFLAUNCH_IMAGE_STEPS", "20"))
pipe = AutoPipelineForText2Image.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    use_safetensors=True,
).to("cuda")
background = DiffusersBackgroundProvider(
    pipe,
    model_ref=model_id,
    steps=steps,
    guidance_scale=4.0,
    release_after_render=True,
)
semantic_scorer = HuggingFaceSigLIP2Scorer(
    model_id="google/siglip2-base-patch16-224",
    minimum_positive_probability=0.34,
    minimum_margin=0.05,
)

run_label = os.environ.get("PROOFLAUNCH_RUN_LABEL", "frozen-visual-001")
run_dir = radeon_evidence / run_label
run_dir.mkdir(parents=True, exist_ok=True)
product_path = run_dir / "fixture-product.png"
product = Image.new("RGBA", (720, 900), (0, 0, 0, 0))
draw = ImageDraw.Draw(product)
draw.rounded_rectangle(
    (95, 55, 625, 835), radius=90,
    fill=(245, 248, 255, 255), outline=(174, 201, 239, 255), width=12,
)
draw.rounded_rectangle((185, 180, 535, 565), radius=36, fill=(16, 74, 154, 255))
draw.ellipse((275, 655, 445, 825), fill=(255, 187, 76, 255))
product.save(product_path, format="PNG", optimize=False)

request = CampaignRequest(
    product_name="Aster Arc",
    product_image=str(product_path),
    specification_text="Battery life up to 18 hours under the documented test.",
    market="Japan",
    platform="TikTok",
    goal="launch",
    target_emotion="joy",
)
supervisor = build_supervisor(
    background=background,
    audience=recorded_audience,
    seed=202607,
    canvas_overrides={"TikTok": (768, 1344)},
    unsafe_first_pass=True,
    semantic_scorer=semantic_scorer,
)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
state = supervisor.run(request, str(run_dir), run_id="radeon-frozen-visual-001")
torch.cuda.synchronize()
latency_seconds = time.perf_counter() - started
peak_vram_bytes = torch.cuda.max_memory_allocated()
semantic_scorer.release()

assert state.stage is Stage.DELIVER, state.blocked_reason
assert not state.release_blockers, state.release_blockers
assert state.repair_count <= 2
assert state.assets[0].metadata["semantic_score"]["passed"]
assert EvidenceLedger.verify(run_dir / "evidence-ledger.jsonl")
visual_evidence = {
    "schema": "ProofLaunchRadeonVisualRunV1",
    "runtime": runtime,
    "execution_plan": execution_plan,
    "model_id": model_id,
    "steps": steps,
    "seed": 202607,
    "canvas": [768, 1344],
    "stage": state.stage.value,
    "repair_count": state.repair_count,
    "latency_seconds": latency_seconds,
    "peak_vram_bytes": peak_vram_bytes,
    "asset_path": state.assets[0].path,
    "asset_sha256": sha256_file(state.assets[0].path),
    "product_source_sha256": sha256_file(product_path),
    "protected_layer_sha256": state.assets[0].metadata["product_layer_sha256"],
    "semantic_score": state.assets[0].metadata.get("semantic_score"),
    "backend_kind": state.assets[0].metadata["backend_kind"],
    "exports": [
        {"path": path, "sha256": sha256_file(path)} for path in state.exports
    ],
}
(radeon_evidence / "generation-benchmark.json").write_text(
    json.dumps(visual_evidence, indent=2), encoding="utf-8"
)
print(json.dumps(visual_evidence, indent=2))



## 5B. Generate a short local video with official LTX-Video 2B on Radeon

The 2B distilled checkpoint is the 24 GB-safe required tier. A separately
declared LTX-2.3 22B quality tier may be benchmarked only when the instance has
at least 46 GB and its ROCm patch is recorded. This cell uses the official
Lightricks inference library and records the exact Git revision and checkpoint.
It then overlays the owned product layer after generation on every frame.


In [ ]:

import imageio.v2 as imageio
from PIL import Image, ImageDraw

ltx_root = repo / ".vendor" / "LTX-Video"
if not (ltx_root / ".git").is_dir():
    ltx_root.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/Lightricks/LTX-Video.git",
            str(ltx_root),
        ],
        check=True,
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{ltx_root}[inference]"],
    check=True,
)
ltx_revision = subprocess.check_output(
    ["git", "-C", str(ltx_root), "rev-parse", "HEAD"],
    text=True,
).strip()
sys.path.insert(0, str(ltx_root))
from ltx_video.inference import InferenceConfig, infer
from prooflaunch_remix.video_generation import IdentityLockedVideoComposer

ltx_output_dir = multimodal_run_dir / "ltx-video"
ltx_output_dir.mkdir(parents=True, exist_ok=True)
before_videos = set(ltx_output_dir.glob("*.mp4"))
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
video_started = time.perf_counter()
infer(
    InferenceConfig(
        pipeline_config=str(
            ltx_root / "configs" / "ltxv-2b-0.9.8-distilled.yaml"
        ),
        prompt=(
            "A precise studio product shot. The camera makes a slow dolly-in. "
            "Soft blue and amber light moves across a clean dark background. "
            "The supplied product remains centered, rigid, and unchanged. "
            "No text, no extra objects, no logo mutation."
        ),
        output_path=str(ltx_output_dir),
        seed=202607,
        height=512,
        width=768,
        num_frames=97,
        frame_rate=24,
        offload_to_cpu=available_vram_gb < 30,
        conditioning_media_paths=[str(mission_product_path)],
        conditioning_strengths=[1.0],
        conditioning_start_frames=[0],
    )
)
torch.cuda.synchronize()
video_latency_seconds = time.perf_counter() - video_started
video_peak_vram_bytes = torch.cuda.max_memory_allocated()
new_videos = set(ltx_output_dir.glob("*.mp4")) - before_videos
assert new_videos, "LTX-Video did not create a new MP4"
raw_video_path = max(new_videos, key=lambda item: item.stat().st_mtime_ns)

reader = imageio.get_reader(raw_video_path)
raw_frames = tuple(Image.fromarray(frame).convert("RGBA") for frame in reader)
reader.close()
locked_frames, identity_report = IdentityLockedVideoComposer().compose(
    generated_frames=raw_frames,
    product_layer=Image.open(mission_product_path).convert("RGBA"),
    normalized_box=(0.32, 0.28, 0.68, 0.78),
)
assert identity_report.passed, identity_report
# Add a measurable visual launch cue outside the protected product box.
cued_frames = []
for frame_index, frame in enumerate(locked_frames):
    rendered = frame.copy()
    if 24 <= frame_index <= 31:
        cue = ImageDraw.Draw(rendered)
        radius = 20 + 3 * (frame_index - 24)
        cue.ellipse(
            (54 - radius, 54 - radius, 54 + radius, 54 + radius),
            outline=(255, 187, 76, 255), width=8,
        )
    cued_frames.append(rendered)
locked_frames = tuple(cued_frames)
locked_video_path = ltx_output_dir / "prooflaunch-identity-locked.mp4"
with imageio.get_writer(
    locked_video_path,
    fps=24,
    codec="libx264",
    quality=10,
) as writer:
    for frame in locked_frames:
        writer.append_data(frame.convert("RGB"))
assert locked_video_path.is_file() and locked_video_path.stat().st_size > 0

video_evidence = {
    "schema": "ProofLaunchRadeonVideoRunV2",
    "runtime": runtime,
    "model": "Lightricks/LTX-Video:ltxv-2b-0.9.8-distilled.safetensors",
    "model_license": "LTX-Video Open Weights License 1.0",
    "official_code_revision": ltx_revision,
    "seed": 202607,
    "frame_count": identity_report.frame_count,
    "fps": 24,
    "latency_seconds": video_latency_seconds,
    "peak_vram_bytes": video_peak_vram_bytes,
    "raw_video_path": str(raw_video_path),
    "raw_video_sha256": sha256_file(raw_video_path),
    "locked_video_path": str(locked_video_path),
    "locked_video_sha256": sha256_file(locked_video_path),
    "pre_encode_identity_report": identity_report.__dict__,
    "truth_boundary": (
        "The pre-encode gate is exact. The following V6 delivery cell must "
        "also pass decoded H.264 identity, SAM2, CLAP, and A/V alignment."
    ),
}
(multimodal_run_dir / "video-benchmark.json").write_text(
    json.dumps(video_evidence, indent=2), encoding="utf-8"
)
print(json.dumps(video_evidence, indent=2))

gc.collect()
torch.cuda.empty_cache()



## 5C. Track the grounded product through every generated frame with SAM2

The GroundingDINO box is mapped through the product compositing transform and
used as SAM2's first-frame prompt. A run passes only when at least 95% of
expected frames are returned and at least 95% contain a non-empty product mask.
SAM2 is unloaded before the audio-text model is loaded.


In [ ]:

from prooflaunch_remix.multimodal_models import LocalSAM2VideoTracker

source_width, source_height = Image.open(mission_product_path).size
video_width, video_height = locked_frames[0].size
container_box = (
    0.32 * video_width,
    0.28 * video_height,
    0.68 * video_width,
    0.78 * video_height,
)
dx1, dy1, dx2, dy2 = dino_detection.box_xyxy
container_width = container_box[2] - container_box[0]
container_height = container_box[3] - container_box[1]
sam_prompt_box = (
    container_box[0] + dx1 / source_width * container_width,
    container_box[1] + dy1 / source_height * container_height,
    container_box[0] + dx2 / source_width * container_width,
    container_box[1] + dy2 / source_height * container_height,
)
sam_tracker = LocalSAM2VideoTracker(
    model_id="facebook/sam2.1-hiera-small",
    minimum_frame_coverage=0.95,
    minimum_nonempty_mask_rate=0.95,
)
sam_report = sam_tracker.track(
    frames=locked_frames,
    prompt_box_xyxy=sam_prompt_box,
    source_artifact_id=product_artifact.id,
)
assert sam_report.passed, sam_report
(multimodal_run_dir / "sam2-temporal-segmentation.json").write_text(
    json.dumps(sam_report, default=lambda item: item.__dict__, indent=2),
    encoding="utf-8",
)
print(json.dumps(sam_report, default=lambda item: item.__dict__, indent=2))

del sam_tracker
gc.collect()
torch.cuda.empty_cache()
print("SAM2 phase complete; Radeon memory released")



## 5D. Produce the final A/V MP4, then audit decoded identity and timing

This cell creates a deterministic owned launch chime, muxes it into the local
video, decodes the actual delivery MP4, and applies four independent checks:
H.264 product identity, visible cue detection, CLAP audio semantics, and
timestamp alignment. Planned timestamps alone cannot pass this gate.


In [ ]:

import math
import wave
import imageio_ffmpeg
import numpy as np

from prooflaunch_remix.media_quality import (
    AudioVideoAlignmentGate,
    EncodedVideoIdentityGate,
    SemanticEvent,
)
from prooflaunch_remix.multimodal_models import LocalCLAPScorer

fps = 24
sample_rate = 48_000
duration_seconds = len(locked_frames) / fps
sample_count = round(duration_seconds * sample_rate)
timeline = np.arange(sample_count, dtype=np.float32) / sample_rate
audio = np.zeros(sample_count, dtype=np.float32)
cue_start, cue_end = 1.0, 1.34
cue_mask = (timeline >= cue_start) & (timeline <= cue_end)
cue_time = timeline[cue_mask] - cue_start
envelope = np.sin(np.pi * cue_time / (cue_end - cue_start)) ** 2
audio[cue_mask] = 0.40 * envelope * (
    np.sin(2 * np.pi * 659.25 * cue_time)
    + 0.55 * np.sin(2 * np.pi * 987.77 * cue_time)
)
audio = np.clip(audio, -0.95, 0.95)
source_audio_path = ltx_output_dir / "owned-launch-chime.wav"
with wave.open(str(source_audio_path), "wb") as handle:
    handle.setnchannels(1)
    handle.setsampwidth(2)
    handle.setframerate(sample_rate)
    handle.writeframes((audio * 32767).astype("<i2").tobytes())

ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
final_video_path = ltx_output_dir / "prooflaunch-final-av.mp4"
subprocess.run(
    [
        ffmpeg, "-y",
        "-i", str(locked_video_path),
        "-i", str(source_audio_path),
        "-c:v", "copy",
        "-c:a", "aac",
        "-b:a", "192k",
        "-shortest",
        str(final_video_path),
    ],
    check=True,
    capture_output=True,
)
assert final_video_path.is_file() and final_video_path.stat().st_size > 0

encoded_identity = EncodedVideoIdentityGate(
    codec_tolerance=0.04,
    minimum_frame_coverage=0.95,
).audit(
    video_path=final_video_path,
    product_layer=Image.open(mission_product_path).convert("RGBA"),
    normalized_box=(0.32, 0.28, 0.68, 0.78),
    expected_frame_count=len(locked_frames),
)
assert encoded_identity.passed, encoded_identity

final_reader = imageio.get_reader(final_video_path)
decoded_frames = tuple(Image.fromarray(frame).convert("RGB") for frame in final_reader)
final_reader.close()
cue_frames = []
for frame_index, frame in enumerate(decoded_frames):
    crop = np.asarray(frame.crop((5, 5, 103, 103)))
    amber = (
        (crop[:, :, 0] > 180)
        & (crop[:, :, 1] > 100)
        & (crop[:, :, 2] < 150)
    )
    if int(amber.sum()) >= 35:
        cue_frames.append(frame_index)
assert cue_frames, "the encoded visual cue was not detected"
video_event = SemanticEvent(
    "launch chime",
    min(cue_frames) / fps,
    (max(cue_frames) + 1) / fps,
    1.0,
    f"decoded_frames:{min(cue_frames)}-{max(cue_frames)}",
)

decoded_audio_path = ltx_output_dir / "decoded-delivery-audio.wav"
subprocess.run(
    [
        ffmpeg, "-y", "-i", str(final_video_path),
        "-vn", "-ac", "1", "-ar", str(sample_rate),
        str(decoded_audio_path),
    ],
    check=True,
    capture_output=True,
)
with wave.open(str(decoded_audio_path), "rb") as handle:
    decoded_audio = np.frombuffer(
        handle.readframes(handle.getnframes()),
        dtype="<i2",
    ).astype(np.float32) / 32768.0
    decoded_audio_duration = handle.getnframes() / handle.getframerate()

clap = LocalCLAPScorer(
    model_id="laion/clap-htsat-fused",
    minimum_probability=0.45,
)
clap_score = clap.score_array(
    audio=decoded_audio,
    sampling_rate=sample_rate,
    expected_label="a bright electronic launch chime",
    candidate_labels=(
        "a bright electronic launch chime",
        "spoken human voice",
        "traffic noise",
        "silence",
    ),
)
assert clap_score.passed, clap_score
audio_event = SemanticEvent(
    "launch chime",
    cue_start,
    cue_end,
    clap_score.expected_probability,
    f"decoded_audio_seconds:{cue_start:.2f}-{cue_end:.2f}",
)
alignment_report = AudioVideoAlignmentGate(
    tolerance_seconds=0.50,
    duration_tolerance_seconds=0.25,
    minimum_event_coverage=0.95,
).audit(
    audio_events=(audio_event,),
    video_events=(video_event,),
    audio_duration_seconds=decoded_audio_duration,
    video_duration_seconds=len(decoded_frames) / fps,
)
assert alignment_report.passed, alignment_report

delivery_evidence = {
    "schema": "ProofLaunchEncodedDeliveryGateV1",
    "final_video_path": str(final_video_path),
    "final_video_sha256": sha256_file(final_video_path),
    "encoded_identity": encoded_identity.to_dict(),
    "clap": clap_score.__dict__,
    "audio_video_alignment": alignment_report.to_dict(),
}
(multimodal_run_dir / "encoded-delivery-gate.json").write_text(
    json.dumps(delivery_evidence, indent=2), encoding="utf-8"
)
print(json.dumps(delivery_evidence, indent=2))

del clap
gc.collect()
torch.cuda.empty_cache()
print("CLAP phase complete; Radeon memory released")



## 5F. Cross-media release gate

Every deliverable claim must point to a Qwen bounding box/frame locator or an
exact PDF page. Unknown rights, missing source IDs, or absent locators block the
pack. Video and image outputs stay in review until a human approves release.


In [ ]:

from prooflaunch_remix.cross_media import (
    CrossMediaReleaseGate,
    DeliverableClaim,
)

all_observations = (
    tuple(qwen_understanding.observations)
    + tuple(pdf_observations)
    + tuple(dino_observations)
)
claim_observation = next(
    item for item in pdf_observations if "18 hours" in item.claim
)
cross_media_findings = CrossMediaReleaseGate().audit(
    artifacts=mission_artifacts,
    observations=all_observations,
    claims=(
        DeliverableClaim(
            deliverable_id="social-image",
            text="Battery life up to 18 hours under the documented test.",
            observation_ids=(claim_observation.id,),
        ),
        DeliverableClaim(
            deliverable_id="short-video",
            text="Battery life up to 18 hours under the documented test.",
            observation_ids=(claim_observation.id,),
        ),
    ),
)
assert not cross_media_findings, cross_media_findings
cross_media_evidence = {
    "schema": "ProofLaunchCrossMediaGateV1",
    "finding_count": len(cross_media_findings),
    "passed": not cross_media_findings,
    "human_release_required": True,
    "mission_graph_hash": mission_plan.graph_hash,
}
(multimodal_run_dir / "cross-media-gate.json").write_text(
    json.dumps(cross_media_evidence, indent=2), encoding="utf-8"
)
print(json.dumps(cross_media_evidence, indent=2))



## 5G. Apply the precommitted competition-2X acceptance contract

One successful fixture is evidence of operability, not evidence of benchmark
quality. This cell proves that the validator rejects the fixture-only record.
If a measured `competition-2x-benchmark.json` exists, it must satisfy every
metric—including raw blind-comparison counts and the Wilson lower bound—before
the submission gate can become ready.


In [ ]:

from prooflaunch_remix.quality_contract import QualityTier, contract_for

fixture_only_evidence = {
    "product_count": 1,
    "aspect_ratio_count": 2,
    "seeds_per_product": 1,
    "human_raters": 0,
    "repair_budget_max": 2,
    "unsupported_claim_count": 0,
    "locator_coverage": 1.0,
    "image_identity_pass_rate": 1.0,
    "preencode_video_identity_pass_rate": 1.0,
    "encoded_video_identity_pass_rate": float(encoded_identity.passed),
    "audio_video_alignment_pass_rate": float(alignment_report.passed),
    "cross_media_pass_rate": float(cross_media_evidence["passed"]),
    "completion_rate": 1.0,
    "crash_rate": 0.0,
    "publish_preferred_count": 0,
    "publish_comparison_count": 0,
}
contract = contract_for(QualityTier.COMPETITION_2X)
fixture_report = contract.evaluate(fixture_only_evidence)
assert not fixture_report.passed, (
    "A single fixture must never certify the competition-2X benchmark"
)
(multimodal_run_dir / "fixture-only-2x-readiness.json").write_text(
    json.dumps(fixture_report.to_dict(), indent=2), encoding="utf-8"
)

benchmark_path = multimodal_run_dir / "competition-2x-benchmark.json"
if benchmark_path.is_file():
    benchmark_evidence = json.loads(benchmark_path.read_text(encoding="utf-8"))
    benchmark_report = contract.evaluate(benchmark_evidence)
    (multimodal_run_dir / "competition-2x-report.json").write_text(
        json.dumps(benchmark_report.to_dict(), indent=2), encoding="utf-8"
    )
    assert benchmark_report.passed, benchmark_report.to_dict()
    print("competition-2X benchmark: PASS")
else:
    benchmark_report = None
    print(
        "competition-2X benchmark: BLOCKED — run 30 products × 3 ratios × "
        "5 seeds and 150+ blind human comparisons"
    )
print(json.dumps(fixture_report.to_dict(), indent=2))



## 6. Submission handoff

Do not change `SUBMISSION_GATE.json` to ready from this one fixture. Complete the
30-product / three-ratio / five-seed benchmark, blinded human comparison,
3–5 minute real-operation video, and final English PR checklist first.
